# **Implementing a Character-Based Tokenizer**

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import requests


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [23]:
import requests

url = 'https://www.gutenberg.org/cache/epub/100/pg100.txt'

response = requests.get(url)
response.raise_for_status()
text = response.text

#print(text[0:1000])

In [24]:
# Building the Tokenizer

class CharTokenizer:
  def __init__(self, vocabulary):
    self.token_id_for_char = {char: token_id for token_id, char in enumerate(vocabulary)}
    self.char_for_token_id = {token_id: char for token_id, char in enumerate(vocabulary)}

  @staticmethod
  def train_from_text(text):
    vocabulary = set(text)
    return CharTokenizer(sorted(list(vocabulary)))

  def encode(self, text):
    token_ids = []
    for char in text:
      token_ids.append(self.token_id_for_char[char])
    return torch.tensor(token_ids, dtype=torch.long)

  def decode(self, token_ids):
    chars = []
    for token_id in token_ids.tolist():
      chars.append(self.char_for_token_id[token_id])
    return ''.join(chars)

  def vocabulary_size(self):
    return len(self.token_id_for_char)

In [25]:
tokenizer = CharTokenizer.train_from_text(text)


In [26]:
print(tokenizer.encode("Hello world"))


tensor([38, 64, 71, 71, 74,  3, 82, 74, 77, 71, 63])


In [27]:
print(tokenizer.decode(tokenizer.encode("Hello world")))


Hello world


In [28]:
tokenizer.vocabulary_size()


108

In [29]:
# Displays the mapping between characters and token IDs.

import pprint
pp = pprint.PrettyPrinter(depth=4)
# pp.pprint(tokenizer.char_for_token_id)
# pp.pprint(tokenizer.token_id_for_char)

# Implementing a Data Loader

In [30]:
from torch.utils.data import Dataset

class TokenIdsDataset(Dataset):
    def __init__(self, data, block_size):

        self.data = data
        self.block_size = block_size

    def __len__(self):

        return len(self.data) - self.block_size

    def __getitem__(self, pos):

        assert pos < len(self.data) - self.block_size

        x = self.data[pos:pos + self.block_size]
        y = self.data[pos + 1:pos + 1 + self.block_size]

        return x, y

In [31]:
# Step 2 - Tokenize the Text

tokenizer_encode = tokenizer.encode(text)
dataset = TokenIdsDataset(tokenizer_encode, block_size=64)

In [32]:
# Step 3 - Retrieve the First Item from the Dataset
x,y = dataset[300]

print(tokenizer.decode(x))

the Project Gutenberg License included with this eBook or online


In [33]:
from torch.utils.data import DataLoader, RandomSampler

# Create random sampler from a datasset
sampler = RandomSampler(dataset, replacement=True)

# Use Dataloader to laod two random samplers using the sampler
dataloader = DataLoader(dataset, batch_size=2, sampler=sampler)

In [38]:
x, y = next(iter(dataloader))
print(x.shape)
print(x)

torch.Size([2, 64])
tensor([[ 68,  73,   3,  79,  67,  64,   3,  46,  60,  71,  60,  62,  64,  16,
           2,   1,  49,  62,  64,  73,  64,   3,  39,  39,  39,  16,   3,  35,
          60,  78,  79,  62,  67,  64,  60,  75,  16,   3,  31,   3,  48,  74,
          74,  72,   3,  68,  73,   3,  79,  67,  64,   3,  32,  74,  60,  77,
         102,  78,   3,  38,  64,  60,  63,   3],
        [ 74,  68,  73,  79,  16,   3,  49,  75,  64,  60,  70,  14,   3,  72,
          60,  73,  29,   3,  79,  67,  84,   3,  79,  74,  73,  66,  80,  64,
           2,   1,  43,  60,  84,   3,  79,  60,  70,  64,   3,  74,  65,  65,
           3,  78,  74,  72,  64,   3,  64,  83,  79,  77,  64,  72,  68,  79,
          84,  14,   3,  82,  67,  68,  62,  67]])


In [41]:
print(tokenizer.decode(x[0]))


in the Palace.
Scene III. Eastcheap. A Room in the Boar’s Head 


In [42]:
print(tokenizer.decode(y[0]))

n the Palace.
Scene III. Eastcheap. A Room in the Boar’s Head T
